# Level 7: Quantum Fourier Transform (QFT)
## Q# Edition

In this notebook, you will:

1. Build QFT from scratch in Q#
2. Use Q#'s built-in `ApplyQFT` from the standard library
3. Understand controlled phase rotations in Q#

In [ ]:
import qsharp
from qsharp_widgets import Circuit
print("Q# ready!")

---
## 1. QFT Building Blocks

The QFT transforms computational basis states into phase-encoded states:

$$\text{QFT}|j\rangle = \frac{1}{\sqrt{N}} \sum_{k=0}^{N-1} e^{2\pi i jk/N} |k\rangle$$

The circuit uses H gates and controlled-$R_k$ rotations where $R_k$ rotates by $2\pi/2^k$.

In [ ]:
%%qsharp

open Microsoft.Quantum.Math;
open Microsoft.Quantum.Diagnostics;

// Manual QFT implementation
operation MyQFT(qs : Qubit[]) : Unit is Adj {
    let n = Length(qs);

    for i in 0..n - 1 {
        H(qs[i]);

        for j in i + 1..n - 1 {
            let k = j - i + 1;
            let angle = 2.0 * PI() / IntAsDouble(1 <<< k);
            Controlled R1([qs[j]], (angle, qs[i]));
        }
    }

    // Reverse qubit order
    for i in 0..n / 2 - 1 {
        SWAP(qs[i], qs[n - 1 - i]);
    }
}

In [ ]:
%%qsharp

// Apply QFT to |000> and inspect the state
operation QFTDemo() : Unit {
    use qs = Qubit[3];

    Message("Before QFT (state |000>):");
    DumpMachine();

    MyQFT(qs);

    Message("\nAfter QFT:");
    DumpMachine();

    ResetAll(qs);
}

In [ ]:
qsharp.eval("QFTDemo()")

In [ ]:
# Visualize the QFT circuit
Circuit(qsharp.circuit("QFTDemo()"))

---
## 2. QFT on Different Input States

In [ ]:
%%qsharp

// QFT on |101>
operation QFT_On_101() : Unit {
    use qs = Qubit[3];

    X(qs[0]);
    X(qs[2]);

    Message("Before QFT (state |101>):");
    DumpMachine();

    MyQFT(qs);

    Message("\nAfter QFT:");
    DumpMachine();

    ResetAll(qs);
}

In [ ]:
qsharp.eval("QFT_On_101()")
print("\nNotice: All probabilities are equal (1/8) — the information is in the phases!")

---
## 3. Inverse QFT

Q# makes this trivial with the `Adj` functor — `Adjoint MyQFT` automatically gives QFT†.

In [ ]:
%%qsharp

// Verify QFT followed by QFT† recovers the original state
operation QFTRoundTrip() : Result[] {
    use qs = Qubit[3];

    // Start with |101>
    X(qs[0]);
    X(qs[2]);

    // Apply QFT then inverse QFT
    MyQFT(qs);
    Adjoint MyQFT(qs);

    Message("After QFT → QFT† (should be |101>):");
    DumpMachine();

    let results = MeasureEachZ(qs);
    ResetAll(qs);
    return results;
}

In [ ]:
result = qsharp.eval("QFTRoundTrip()")
bits = ''.join(['1' if r == 'One' else '0' for r in result])
print(f"\nRecovered: |{bits}> ✓")

---
## 4. Using the Standard Library QFT

Q# provides `ApplyQFT` in the standard library — let's verify it matches our implementation.

In [ ]:
%%qsharp

open Microsoft.Quantum.Canon;

operation LibraryQFTDemo() : Unit {
    use qs = Qubit[3];

    X(qs[0]);
    X(qs[2]);

    Message("Using standard library ApplyQFT on |101>:");
    ApplyQFT(qs);
    DumpMachine();

    ResetAll(qs);
}

In [ ]:
qsharp.eval("LibraryQFTDemo()")
print("\nSame result as our manual implementation!")

---
## 5. Statistical Measurement After QFT

In [ ]:
%%qsharp

operation QFTMeasureMany(shots : Int) : Int[] {
    mutable counts = [0, size = 8];

    for _ in 0..shots - 1 {
        use qs = Qubit[3];
        X(qs[0]);
        X(qs[2]);
        MyQFT(qs);

        let r = MeasureEachZ(qs);
        mutable idx = 0;
        for i in 0..2 {
            if r[i] == One { set idx = idx + (1 <<< i); }
        }
        set counts w/= idx <- counts[idx] + 1;

        ResetAll(qs);
    }

    return counts;
}

In [ ]:
import matplotlib.pyplot as plt

counts = qsharp.eval("QFTMeasureMany(10000)")
labels = [f"|{i:03b}>" for i in range(8)]

plt.figure(figsize=(10, 5))
plt.bar(labels, counts, color='darkorange')
plt.xlabel('State')
plt.ylabel('Count')
plt.title('QFT|101>: Measurement Distribution (10,000 shots)')
plt.axhline(y=10000/8, color='r', linestyle='--', label='Uniform (1250)')
plt.legend()
plt.show()
print("Uniform distribution confirms phase-only encoding!")

---
## 6. Key Takeaways

| Concept | Description |
|---------|-------------|
| **QFT** | Converts computational to frequency basis |
| **$O(n^2)$ gates** | Exponentially faster than classical FFT |
| **Adj functor** | `Adjoint MyQFT` gives inverse QFT for free |
| **ApplyQFT** | Q# standard library implementation |

---
## 7. Challenges

1. **5-qubit QFT**: Run `MyQFT` on 5 qubits with state |10101>. Verify round-trip.
2. **Compare implementations**: Compare `MyQFT` and `ApplyQFT` outputs on the same input.
3. **QFT addition**: Use QFT to add two 2-qubit numbers in the Fourier basis.

In [ ]:
%%qsharp

// Your challenge code here!

---
**Next up: [Level 8 — Phase Estimation & Shor's Algorithm](../Level_08_Phase_Estimation_Shor/Level_08_QSharp.ipynb)**